In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("Telco_customer_churn.xlsx")


In [ ]:
df['Churn Label'].value_counts()


,count
Churn Label,
No,5174
Yes,1869


In [ ]:
df['Total Charges'].dtype


dtype('O')

In [ ]:
(df['Total Charges'] == " ").sum()


np.int64(11)

In [ ]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')


In [ ]:
df['Total Charges'].isnull().sum()


np.int64(11)

In [ ]:
df['Total Charges'].fillna(df['Total Charges'].median(), inplace=True)


/tmp/ipython-input-2939802501.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Total Charges'].fillna(df['Total Charges'].median(), inplace=True)


In [ ]:
df['Churn Label'] = df['Churn Label'].map({'No': 0, 'Yes': 1})


In [ ]:
df['Churn Label'].value_counts()


,count
Churn Label,
0,5174
1,1869


In [ ]:
df.drop(columns=['CustomerID'], inplace=True)


In [ ]:
X = df.drop('Churn Label', axis=1)
y = df['Churn Label']


In [ ]:
X = pd.get_dummies(X, drop_first=True)


In [ ]:
X.shape, y.shape


((7043, 2835), (7043,))

In [ ]:
X.isnull().sum().sum()


np.int64(0)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn Label', axis=1)
y = df['Churn Label']
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [ ]:
y_pred_lr = lr.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1]))


Accuracy: 1.0
              precision    recall  f1-score   support

          No       1.00      1.00      1.00      1035
         Yes       1.00      1.00      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409

ROC-AUC: 1.0


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)


RandomForestClassifier(n_estimators=200, random_state=42)

In [ ]:
y_pred_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))


Accuracy: 0.9978708303761533
              precision    recall  f1-score   support

          No       1.00      1.00      1.00      1035
         Yes       1.00      0.99      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409

ROC-AUC: 0.9999974166214576


### Model Performance Comparison

The performance of two baseline models was evaluated using Accuracy, Precision, Recall, F1-score, and ROC-AUC on the test dataset.

**Logistic Regression Results:**
- Accuracy: 1.00
- Precision (Churn = Yes): 1.00
- Recall (Churn = Yes): 1.00
- F1-score (Churn = Yes): 1.00
- ROC-AUC Score: 1.00

Logistic Regression achieved perfect performance on the test set, correctly classifying all churned and non-churned customers.

**Random Forest Results:**
- Accuracy: 0.998
- Precision (Churn = Yes): 1.00
- Recall (Churn = Yes): 0.99
- F1-score (Churn = Yes): 1.00
- ROC-AUC Score: 0.9999

Random Forest also performed extremely well, with only a very small number of misclassifications compared to Logistic Regression.

**Summary:**
Both models demonstrated excellent performance on the test data. Logistic Regression slightly outperformed Random Forest in terms of recall and ROC-AUC, while also offering better interpretability. Therefore, Logistic Regression can be considered a strong baseline model for this churn prediction problem.
